In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -q \
  transformers==4.48.3 \
  torch \
  peft==0.14.0 \
  trl==0.15.2 \
  faiss-cpu \
  sentence-transformers==3.3.1 \
  mutmut \
  pandas \
  accelerate==1.2.1 \
  datasets \
  --break-system-packages

In [ ]:
# 1. Enter the folder
%cd /content/drive/MyDrive/Capstone/oneiros

# 2. Update the script to disable the mock generator
!sed -i 's/use_mock_generator=True/use_mock_generator=False/' oneiros_loop.py

/content/drive/MyDrive/Capstone/oneiros


In [ ]:
%%writefile /content/drive/MyDrive/Capstone/oneiros/harness/execution_harness.py
"""
Execution Harness for running test cases against functions.

This module handles test execution, result collection, and
differential testing between golden and mutant functions.
"""
import sys
import ast
import traceback
import time
import signal
import multiprocessing
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Callable
from dataclasses import dataclass, field, asdict
from enum import Enum
import io
from contextlib import redirect_stdout, redirect_stderr

sys.path.insert(0, str(Path(__file__).parent.parent))

from harness.dataset_loader import TargetFunction
from harness.mutation_engine import Mutant


class TestResult(Enum):
    """Possible outcomes of test execution."""
    PASS = "pass"                    # Test passed (no bug found)
    FAIL = "fail"                    # Test found a bug (assertion failed)
    ERROR = "error"                  # Test caused an error
    TIMEOUT = "timeout"              # Test timed out
    CRASH = "crash"                  # Test caused a crash


@dataclass
class ExecutionResult:
    """Result of executing a test case."""
    test_id: str
    target_id: str
    result: TestResult
    output: str = ""
    error_message: str = ""
    execution_time: float = 0.0

    # Differential testing results
    golden_output: Any = None
    mutant_output: Any = None
    outputs_differ: bool = False

    def to_dict(self) -> Dict[str, Any]:
        data = asdict(self)
        data["result"] = self.result.value
        return data

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "ExecutionResult":
        data["result"] = TestResult(data["result"])
        return cls(**data)

    def is_bug_found(self) -> bool:
        """Check if this result indicates a bug was found."""
        return self.result in (TestResult.FAIL, TestResult.ERROR) or self.outputs_differ


@dataclass
class TestCase:
    """Represents a test case to execute."""
    id: str
    code: str                        # The test code to run
    target_function: str             # Name of function being tested
    inputs: Dict[str, Any] = field(default_factory=dict)  # Test inputs
    expected_output: Any = None      # Expected output (if known)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class ExecutionHarness:
    """
    Harness for executing test cases against Python functions.
    """

    def __init__(self, timeout_seconds: float = 5.0):
        """
        Initialize the execution harness.

        Args:
            timeout_seconds: Maximum time for test execution
        """
        self.timeout = timeout_seconds
        self.execution_count = 0

    def _create_execution_namespace(
        self,
        function_code: str
    ) -> Dict[str, Any]:
        """
        Create a clean namespace with the function defined.

        Args:
            function_code: The function code to execute

        Returns:
            Namespace dictionary
        """
        namespace = {
            '__builtins__': __builtins__,
            'List': List,
            'Dict': Dict,
            'Any': Any,
            'Optional': Optional,
            'Tuple': Tuple,
        }

        try:
            exec(function_code, namespace)
        except Exception as e:
            raise RuntimeError(f"Failed to define function: {e}")

        return namespace

    def execute_test(
        self,
        test_code: str,
        function_code: str,
        entry_point: str,
        test_id: str = None
    ) -> ExecutionResult:
        """
        Execute a single test against a function.

        Args:
            test_code: The test code to run
            function_code: The function being tested
            entry_point: Name of the function to call
            test_id: Identifier for the test

        Returns:
            ExecutionResult with the outcome
        """
        test_id = test_id or f"test_{self.execution_count}"
        self.execution_count += 1

        start_time = time.time()
        stdout_capture = io.StringIO()
        stderr_capture = io.StringIO()

        try:
            # Create namespace with function
            namespace = self._create_execution_namespace(function_code)

            # Execute test code
            with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
                exec(test_code, namespace)

            execution_time = time.time() - start_time

            return ExecutionResult(
                test_id=test_id,
                target_id=entry_point,
                result=TestResult.PASS,
                output=stdout_capture.getvalue(),
                execution_time=execution_time
            )

        except AssertionError as e:
            execution_time = time.time() - start_time
            return ExecutionResult(
                test_id=test_id,
                target_id=entry_point,
                result=TestResult.FAIL,
                output=stdout_capture.getvalue(),
                error_message=str(e) or "Assertion failed",
                execution_time=execution_time
            )

        except Exception as e:
            execution_time = time.time() - start_time
            return ExecutionResult(
                test_id=test_id,
                target_id=entry_point,
                result=TestResult.ERROR,
                output=stdout_capture.getvalue(),
                error_message=f"{type(e).__name__}: {str(e)}",
                execution_time=execution_time
            )

    def execute_test_batch(
        self,
        tests: List[TestCase],
        function_code: str,
        entry_point: str
    ) -> List[ExecutionResult]:
        """
        Execute a batch of tests against a function.

        Args:
            tests: List of test cases
            function_code: The function being tested
            entry_point: Name of the function to call

        Returns:
            List of ExecutionResults
        """
        results = []

        for test in tests:
            result = self.execute_test(
                test_code=test.code,
                function_code=function_code,
                entry_point=entry_point,
                test_id=test.id
            )
            results.append(result)

        return results

    def differential_test(
        self,
        test_code: str,
        golden_function: TargetFunction,
        mutant: Mutant,
        test_id: str = None
    ) -> ExecutionResult:
        """
        Execute differential test between golden and mutant.

        Args:
            test_code: Test code that exercises the function
            golden_function: The correct function
            mutant: The mutant to test
            test_id: Identifier for the test

        Returns:
            ExecutionResult with differential info
        """
        test_id = test_id or f"diff_test_{self.execution_count}"
        self.execution_count += 1

        start_time = time.time()

        # Execute on golden function
        golden_result = self._safe_execute(
            test_code,
            golden_function.code,
            golden_function.entry_point
        )

        # Execute on mutant
        mutant_result = self._safe_execute(
            test_code,
            mutant.code,
            mutant.entry_point
        )

        execution_time = time.time() - start_time

        # Check if outputs differ
        outputs_differ = False
        if golden_result["success"] and mutant_result["success"]:
            outputs_differ = golden_result["output"] != mutant_result["output"]
        elif golden_result["success"] != mutant_result["success"]:
            outputs_differ = True

        # Determine result
        if outputs_differ:
            result = TestResult.FAIL  # Found a difference (bug exposed)
        elif not mutant_result["success"]:
            result = TestResult.ERROR
        else:
            result = TestResult.PASS

        return ExecutionResult(
            test_id=test_id,
            target_id=mutant.id,
            result=result,
            output=str(mutant_result.get("output", "")),
            error_message=mutant_result.get("error", ""),
            execution_time=execution_time,
            golden_output=golden_result.get("output"),
            mutant_output=mutant_result.get("output"),
            outputs_differ=outputs_differ
        )

    def _safe_execute(
        self,
        test_code: str,
        function_code: str,
        entry_point: str
    ) -> Dict[str, Any]:
        """
        Safely execute code and capture result.

        Returns:
            Dict with success, output, and error fields
        """
        stdout_capture = io.StringIO()

        try:
            namespace = self._create_execution_namespace(function_code)

            with redirect_stdout(stdout_capture):
                exec(test_code, namespace)

            # Try to get the result variable if set
            result = namespace.get('result', namespace.get('output', None))

            return {
                "success": True,
                "output": result if result is not None else stdout_capture.getvalue(),
                "error": None
            }

        except Exception as e:
            return {
                "success": False,
                "output": None,
                "error": f"{type(e).__name__}: {str(e)}"
            }

    def batch_differential_test(
        self,
        tests: List[str],
        golden_function: TargetFunction,
        mutant: Mutant
    ) -> List[ExecutionResult]:
        """
        Run multiple differential tests.

        Args:
            tests: List of test code strings
            golden_function: The correct function
            mutant: The mutant to test

        Returns:
            List of ExecutionResults
        """
        results = []

        for i, test_code in enumerate(tests):
            result = self.differential_test(
                test_code=test_code,
                golden_function=golden_function,
                mutant=mutant,
                test_id=f"batch_diff_{mutant.id}_{i}"
            )
            results.append(result)

        return results

    def label_results(
        self,
        results: List[ExecutionResult]
    ) -> Tuple[List[ExecutionResult], List[ExecutionResult]]:
        """
        Label results as winners (found bugs) and losers (didn't find bugs).

        Args:
            results: List of execution results

        Returns:
            Tuple of (winners, losers)
        """
        winners = []
        losers = []

        for result in results:
            if result.is_bug_found():
                winners.append(result)
            else:
                losers.append(result)

        return winners, losers


class TestGenerator:
    """
    Generates basic test cases for functions.
    Used for seed memory initialization.
    """

    def __init__(self):
        pass

    def generate_simple_tests(
        self,
        function: TargetFunction,
        num_tests: int = 5
    ) -> List[TestCase]:
        """
        Generate simple test cases for a function.

        Args:
            function: The target function
            num_tests: Number of tests to generate

        Returns:
            List of TestCase objects
        """
        tests = []
        # Determine entry point name (TargetFunction uses entry_point, SystemLevelFunction uses name)
        entry_point = getattr(function, 'entry_point', getattr(function, 'name', 'unknown'))

        # Parse function signature for parameters
        params = self._extract_parameters(function.signature)

        # Generate tests based on parameter types
        for i in range(num_tests):
            inputs = self._generate_inputs(params, seed=i)
            test_code = self._create_test_code(
                entry_point,
                inputs
            )

            tests.append(TestCase(
                id=f"seed_test_{function.id}_{i}",
                code=test_code,
                target_function=entry_point,
                inputs=inputs
            ))

        return tests

    def _extract_parameters(self, signature: str) -> List[Dict[str, Any]]:
        """Extract parameter info from signature."""
        params = []

        # Simple regex to find parameters
        import re
        match = re.search(r'\((.*?)\)', signature)
        if not match:
            return params

        param_str = match.group(1)
        if not param_str.strip():
            return params

        for p in param_str.split(','):
            p = p.strip()
            if not p:
                continue

            # Handle type hints
            if ':' in p:
                name, type_hint = p.split(':', 1)
                name = name.strip()
                type_hint = type_hint.split('=')[0].strip()
            else:
                name = p.split('=')[0].strip()
                type_hint = "Any"

            # Infer type from hint
            inferred_type = self._infer_type(type_hint)

            params.append({
                "name": name,
                "type_hint": type_hint,
                "inferred_type": inferred_type
            })

        return params

    def _infer_type(self, type_hint: str) -> str:
        """Infer Python type from type hint."""
        type_hint = type_hint.lower()

        if 'int' in type_hint:
            return 'int'
        elif 'float' in type_hint:
            return 'float'
        elif 'str' in type_hint:
            return 'str'
        elif 'bool' in type_hint:
            return 'bool'
        elif 'list' in type_hint:
            return 'list'
        elif 'dict' in type_hint:
            return 'dict'
        else:
            return 'any'

    def _generate_inputs(
        self,
        params: List[Dict[str, Any]],
        seed: int = 0
    ) -> Dict[str, Any]:
        """Generate input values for parameters."""
        import random
        random.seed(seed)

        inputs = {}

        for param in params:
            name = param["name"]
            ptype = param["inferred_type"]

            if ptype == 'int':
                inputs[name] = random.randint(-10, 100)
            elif ptype == 'float':
                inputs[name] = round(random.uniform(-10, 100), 2)
            elif ptype == 'str':
                strings = ["", "a", "hello", "test123", "  spaces  "]
                inputs[name] = random.choice(strings)
            elif ptype == 'bool':
                inputs[name] = random.choice([True, False])
            elif ptype == 'list':
                inputs[name] = [random.randint(0, 10) for _ in range(random.randint(0, 5))]
            elif ptype == 'dict':
                inputs[name] = {"key": random.randint(0, 10)}
            else:
                inputs[name] = random.randint(0, 10)

        return inputs

    def _create_test_code(
        self,
        entry_point: str,
        inputs: Dict[str, Any]
    ) -> str:
        """Create test code string."""
        # Format arguments
        args = ", ".join(
            f"{k}={repr(v)}" for k, v in inputs.items()
        )

        return f"""
# Auto-generated test case
try:
    result = {entry_point}({args})
except Exception as e:
    result = f"ERROR: {{type(e).__name__}}: {{e}}"
"""


def create_seed_tests(functions: List[TargetFunction]) -> List[TestCase]:
    """
    Create seed test cases for memory initialization.

    Args:
        functions: List of target functions

    Returns:
        List of TestCase objects
    """
    generator = TestGenerator()
    all_tests = []

    for func in functions:
        tests = generator.generate_simple_tests(func, num_tests=3)
        all_tests.extend(tests)

    return all_tests


if __name__ == "__main__":
    print("=" * 60)
    print("Testing Execution Harness")
    print("=" * 60)

    # Test with a simple function
    sample_function = """
def add(a, b):
    \"\"\"Add two numbers.\"\"\"
    return a + b
"""

    # Create a test
    test_code = """
result = add(2, 3)
assert result == 5, f"Expected 5, got {result}"
"""

    harness = ExecutionHarness()

    print("\n1. Testing passing test...")
    result = harness.execute_test(
        test_code=test_code,
        function_code=sample_function,
        entry_point="add",
        test_id="test_add_pass"
    )
    print(f"   Result: {result.result.value}")

    print("\n2. Testing failing test...")
    failing_test = """
result = add(2, 3)
assert result == 6, f"Expected 6, got {result}"
"""
    result = harness.execute_test(
        test_code=failing_test,
        function_code=sample_function,
        entry_point="add",
        test_id="test_add_fail"
    )
    print(f"   Result: {result.result.value}")
    print(f"   Error: {result.error_message}")

    print("\n3. Testing buggy mutant...")
    buggy_function = """
def add(a, b):
    \"\"\"Add two numbers.\"\"\"
    return a - b  # Bug: should be +
"""

    # Create mock objects
    from dataclasses import dataclass

    @dataclass
    class MockGolden:
        code: str = sample_function
        entry_point: str = "add"

    @dataclass
    class MockMutant:
        id: str = "mutant_1"
        code: str = buggy_function
        entry_point: str = "add"

    diff_result = harness.differential_test(
        test_code="result = add(5, 3)",
        golden_function=MockGolden(),
        mutant=MockMutant()
    )
    print(f"   Result: {diff_result.result.value}")
    print(f"   Golden output: {diff_result.golden_output}")
    print(f"   Mutant output: {diff_result.mutant_output}")
    print(f"   Outputs differ: {diff_result.outputs_differ}")

    print("\n" + "=" * 60)
    print("Execution Harness tests complete!")
    print("=" * 60)


Overwriting /content/drive/MyDrive/Capstone/oneiros/harness/execution_harness.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile /content/drive/MyDrive/Capstone/oneiros/engine/generator.py

"""
Phi-3 Generator Module for Oneiros Engine.

This module handles test input generation using the Phi-3-mini-4k-instruct model.
It generates test cases based on function signatures and examples from memory.
"""
import torch
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
import re
import json

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Warning: transformers not installed. Install with: pip install transformers")

try:
    from peft import LoraConfig, get_peft_model, PeftModel
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False
    print("Warning: peft not installed. Install with: pip install peft")

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))

from config import model_config


@dataclass
class GeneratedTest:
    """Represents a generated test case."""
    id: str
    input_code: str              # The generated test input
    function_id: str             # Target function
    raw_output: str              # Raw model output
    is_valid: bool = True        # Whether syntax is valid
    parse_error: str = ""        # Parse error if invalid


class Phi3Generator:
    """
    Test case generator using Phi-3-mini-4k-instruct.

    Generates test inputs for system-level Python functions based on:
    - Function signature and docstring
    - Examples from FAISS memory
    - Edge cases to explore
    """

    def __init__(
        self,
        model_name: str = None,
        load_in_4bit: bool = True,
        device_map: str = "auto"
    ):
        """
        Initialize the Phi-3 generator.

        Args:
            model_name: HuggingFace model name
            load_in_4bit: Whether to use 4-bit quantization (fits on smaller GPUs)
            device_map: Device mapping strategy
        """
        if not TRANSFORMERS_AVAILABLE:
            raise ImportError("transformers is required. Install with: pip install transformers")

        self.model_name = model_name or model_config.model_name
        self.load_in_4bit = load_in_4bit
        self.device_map = device_map

        self.model = None
        self.tokenizer = None
        self.is_loaded = False

        # Generation parameters
        self.max_new_tokens = model_config.max_new_tokens
        self.temperature = model_config.temperature
        self.top_p = model_config.top_p

        # Statistics
        self.stats = {
            "total_generated": 0,
            "valid_generated": 0,
            "invalid_generated": 0
        }

    def load_model(self) -> None:
        """Load the Phi-3 model and tokenizer."""
        if self.is_loaded:
            return

        print(f"Loading {self.model_name}...")

        # Configure quantization
        if self.load_in_4bit:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
        else:
            quantization_config = None

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True
        )

        # Ensure pad token
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load model with RoPE scaling fix
        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(self.model_name, trust_remote_code=True)

        # Patch for RoPE scaling compatibility issues
        if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
            rope_type = config.rope_scaling.get("rope_type", config.rope_scaling.get("type", ""))
            # Phi-3's custom modeling only accepts "su", "yarn", or no rope_scaling
            # "default" / "linear" / etc. are not recognized — just remove it
            if rope_type in ("default", "linear", ""):
                config.rope_scaling = None
            elif "type" not in config.rope_scaling and "rope_type" in config.rope_scaling:
                config.rope_scaling["type"] = config.rope_scaling["rope_type"]

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            config=config,
            quantization_config=quantization_config,
            device_map=self.device_map,
            trust_remote_code=True,
            torch_dtype=torch.float16
        )

        self.is_loaded = True
        print(f"Model loaded successfully!")

    def load_lora_adapter(self, adapter_path: Path) -> None:
        """Load a LoRA adapter (after DPO training)."""
        if not self.is_loaded:
            self.load_model()

        if not PEFT_AVAILABLE:
            raise ImportError("peft is required for LoRA. Install with: pip install peft")

        print(f"Loading LoRA adapter from {adapter_path}...")
        self.model = PeftModel.from_pretrained(self.model, adapter_path)
        print("LoRA adapter loaded!")

    def _create_prompt(
        self,
        function_signature: str,
        docstring: str,
        edge_cases: List[str],
        memory_examples: List[str] = None,
        library: str = "unknown"
    ) -> str:
        """
        Create a prompt for test generation.
        """
        memory_section = ""
        if memory_examples:
            examples = "\n".join(f"- {ex}" for ex in memory_examples[:3])
            memory_section = f"""
Previous successful test inputs:
{examples}
"""

        edge_case_section = ""
        if edge_cases:
            cases = ", ".join(edge_cases[:5])
            edge_case_section = f"Edge cases to consider: {cases}"

        prompt = f"""You are a test case generator for Python's {library} library.

Function: {function_signature}
Description: {docstring}
{edge_case_section}
{memory_section}
Generate a Python test input that could find bugs or edge cases.
Output ONLY the function call, nothing else.

Example format:
result = function_name(arg1, arg2)

Your test input:"""

        return prompt

    def _parse_output(self, output: str, function_id: str) -> GeneratedTest:
        """Parse model output into a test case."""
        # Clean up output
        output = output.strip()

        # Try to extract just the function call
        lines = output.split('\n')
        test_code = None

        for line in lines:
            line = line.strip()
            if line.startswith('result = ') or line.startswith('output = '):
                test_code = line
                break
            elif '(' in line and ')' in line and '=' in line:
                test_code = line
                break

        if not test_code:
            # Use first non-empty line
            for line in lines:
                if line.strip():
                    test_code = line.strip()
                    break

        if not test_code:
            test_code = output[:200]  # Fallback

        # Validate syntax
        is_valid = True
        parse_error = ""
        try:
            compile(test_code, '<string>', 'exec')
        except SyntaxError as e:
            is_valid = False
            parse_error = str(e)

        test_id = f"gen_{function_id}_{self.stats['total_generated']}"
        self.stats["total_generated"] += 1

        if is_valid:
            self.stats["valid_generated"] += 1
        else:
            self.stats["invalid_generated"] += 1

        return GeneratedTest(
            id=test_id,
            input_code=test_code,
            function_id=function_id,
            raw_output=output,
            is_valid=is_valid,
            parse_error=parse_error
        )

    def generate(
        self,
        function_signature: str,
        docstring: str,
        function_id: str,
        edge_cases: List[str] = None,
        memory_examples: List[str] = None,
        library: str = "unknown",
        num_samples: int = 1
    ) -> List[GeneratedTest]:
        """
        Generate test inputs for a function.

        Args:
            function_signature: The function signature
            docstring: Function docstring
            function_id: Unique function ID
            edge_cases: Known edge cases
            memory_examples: Past successful inputs
            library: Library name
            num_samples: Number of tests to generate

        Returns:
            List of GeneratedTest objects
        """
        if not self.is_loaded:
            self.load_model()

        prompt = self._create_prompt(
            function_signature=function_signature,
            docstring=docstring,
            edge_cases=edge_cases or [],
            memory_examples=memory_examples or [],
            library=library
        )

        # Tokenize
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                top_p=self.top_p,
                do_sample=True,
                num_return_sequences=num_samples,
                pad_token_id=self.tokenizer.pad_token_id,
                use_cache=True,                        # <-- keep True
                past_key_values=None,                  # <-- ADD: force fresh cache
            )

        # Decode
        generated_tests = []
        for output in outputs:
            # Get only the new tokens
            new_tokens = output[inputs.input_ids.shape[1]:]
            text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)

            test = self._parse_output(text, function_id)
            generated_tests.append(test)

        return generated_tests

    def generate_batch(
        self,
        functions: List[Dict[str, Any]],
        memory_examples_map: Dict[str, List[str]] = None,
        num_per_function: int = 1
    ) -> Dict[str, List[GeneratedTest]]:
        """
        Generate tests for multiple functions.

        Args:
            functions: List of function configs with signature, docstring, id, etc.
            memory_examples_map: Map of function_id to memory examples
            num_per_function: Tests to generate per function

        Returns:
            Dict mapping function_id to list of GeneratedTest
        """
        memory_examples_map = memory_examples_map or {}
        results = {}

        for func in functions:
            func_id = func["id"]
            tests = self.generate(
                function_signature=func.get("signature", ""),
                docstring=func.get("docstring", ""),
                function_id=func_id,
                edge_cases=func.get("edge_cases", []),
                memory_examples=memory_examples_map.get(func_id, []),
                library=func.get("library", "unknown"),
                num_samples=num_per_function
            )
            results[func_id] = tests

        return results

    def get_stats(self) -> Dict[str, int]:
        """Get generation statistics."""
        return self.stats.copy()


# Mock generator for testing without GPU
class MockGenerator:
    """Mock generator for testing without loading the actual model."""

    def __init__(self):
        self.stats = {"total_generated": 0, "valid_generated": 0, "invalid_generated": 0}

    def load_model(self):
        print("MockGenerator: Model loading skipped")

    def generate(
        self,
        function_signature: str,
        docstring: str,
        function_id: str,
        **kwargs
    ) -> List[GeneratedTest]:
        """Generate mock test cases."""
        mock_tests = [
            f"result = {function_signature.split('(')[0].split()[-1]}({{}})",
            f"result = {function_signature.split('(')[0].split()[-1]}(None)",
            f"result = {function_signature.split('(')[0].split()[-1]}([])",
        ]

        self.stats["total_generated"] += 1
        self.stats["valid_generated"] += 1

        return [GeneratedTest(
            id=f"mock_{function_id}_{i}",
            input_code=test,
            function_id=function_id,
            raw_output=test,
            is_valid=True
        ) for i, test in enumerate(mock_tests[:kwargs.get("num_samples", 1)])]


if __name__ == "__main__":
    print("=" * 60)
    print("Testing Phi-3 Generator (Mock Mode)")
    print("=" * 60)

    # Use mock generator for testing
    generator = MockGenerator()

    print("\n1. Generating test for pandas.merge...")
    tests = generator.generate(
        function_signature="def merge_wrapper(left_df, right_df, on=None, how='inner')",
        docstring="Merge two DataFrames on specified column(s).",
        function_id="sys_pandas_merge",
        edge_cases=["Empty DataFrame", "Mismatched keys"],
        num_samples=1
    )

    for test in tests:
        print(f"   Generated: {test.input_code}")
        print(f"   Valid: {test.is_valid}")

    print("\n2. Stats:")
    stats = generator.stats
    for k, v in stats.items():
        print(f"   {k}: {v}")

    print("\n" + "=" * 60)
    print("Generator test complete!")
    print("=" * 60)


Overwriting /content/drive/MyDrive/Capstone/oneiros/engine/generator.py


In [ ]:
%%writefile /content/drive/MyDrive/Capstone/oneiros/engine/dpo_trainer.py

"""
DPO Trainer Module for Oneiros Engine.

This module implements Direct Preference Optimization (DPO) training
for fine-tuning Phi-3 based on Winner/Loser test pairs.
"""

import warnings
import logging
warnings.filterwarnings("ignore", message=".*Trainer.tokenizer.*")
warnings.filterwarnings("ignore", message=".*processing_class.*")
logging.getLogger("transformers").setLevel(logging.ERROR)

import torch
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import json

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False

try:
    from trl import DPOTrainer as TRLDPOTrainer, DPOConfig
    from datasets import Dataset
    TRL_AVAILABLE = True
except ImportError:
    TRL_AVAILABLE = False
    print("Warning: trl not installed. Install with: pip install trl")

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))

from config import model_config, training_config


@dataclass
class DPODataPoint:
    """A single DPO training data point."""
    prompt: str
    chosen: str
    rejected: str
    function_id: str = ""


class DPOTrainer:
    """
    DPO Trainer for fine-tuning Phi-3 on test generation preferences.

    Uses Winner/Loser pairs from the Feedback Oracle to train the model
    to prefer generating bug-finding and novel tests.
    """

    def __init__(
        self,
        model_name: str = None,
        output_dir: Path = None,
        learning_rate: float = None,
        beta: float = None
    ):
        if not TRL_AVAILABLE:
            raise ImportError("trl is required. Install with: pip install trl")
        if not PEFT_AVAILABLE:
            raise ImportError("peft is required. Install with: pip install peft")

        self.model_name = model_name or model_config.model_name
        self.output_dir = Path(output_dir or training_config.checkpoint_dir)
        self.learning_rate = learning_rate or training_config.learning_rate
        self.beta = beta or training_config.beta

        self.model = None
        self.tokenizer = None
        self.trainer = None

        self.stats = {
            "iterations_completed": 0,
            "total_pairs_trained": 0,
            "best_loss": float("inf")
        }

    def setup_model(self) -> None:
        """Load and prepare model for training."""
        print(f"Setting up {self.model_name} for DPO training...")

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # 4-bit quantization config
        from transformers import BitsAndBytesConfig

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

        # FIX 1: removed undefined `config=`, `quantization_config=` (was wrong var name),
        # and `device_map=self.device_map` (self.device_map never existed)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
            attn_implementation="eager"
        )

        # Prepare for k-bit training
        self.model = prepare_model_for_kbit_training(self.model)

        # CRITICAL FIX for DPO with Gradient Checkpointing
        self.model.enable_input_require_grads()

        # Add LoRA adapters
        checkpoint_path = self.output_dir / "adapter_model.safetensors"
        if checkpoint_path.exists():
            print(f"  Found existing LoRA checkpoint at {self.output_dir}, resuming!")
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(self.model, str(self.output_dir), is_trainable=True)
        else:
            lora_config = LoraConfig(
                r=model_config.lora_r,
                lora_alpha=model_config.lora_alpha,
                lora_dropout=model_config.lora_dropout,
                target_modules=model_config.target_modules,
                bias="none",
                task_type="CAUSAL_LM"
            )
            self.model = get_peft_model(self.model, lora_config)

        print("Model setup complete!")
        # FIX 2: print_trainable_parameters() returns None — don't wrap in print()
        self.model.print_trainable_parameters()

    def prepare_dataset(self, pairs: List[DPODataPoint]) -> 'Dataset':
        """Convert DPO pairs to HuggingFace Dataset."""
        data = {
            "prompt": [p.prompt for p in pairs],
            "chosen": [p.chosen for p in pairs],
            "rejected": [p.rejected for p in pairs]
        }
        return Dataset.from_dict(data)

    def create_prompt(
        self,
        function_signature: str,
        docstring: str,
        library: str = "unknown"
    ) -> str:
        """Create a prompt for the training data."""
        return f"""You are a test case generator for Python's {library} library.

Function: {function_signature}
Description: {docstring}

Generate a Python test input that could find bugs or edge cases.
Output ONLY the function call.

Your test input:"""

    def train(
        self,
        pairs: List[DPODataPoint],
        num_epochs: int = 1,
        batch_size: int = 4
    ) -> Dict[str, Any]:
        """
        Train the model on preference pairs.

        Args:
            pairs: List of DPO training pairs
            num_epochs: Number of training epochs
            batch_size: Batch size for training

        Returns:
            Training results dictionary
        """
        if self.model is None:
            self.setup_model()

        dataset = self.prepare_dataset(pairs)

        # FIX 3: added max_length + max_prompt_length to suppress warnings
        training_args = DPOConfig(
            output_dir=str(self.output_dir),
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            learning_rate=self.learning_rate,
            beta=self.beta,
            logging_steps=10,
            save_steps=100,
            warmup_ratio=0.1,
            fp16=True,
            remove_unused_columns=False,
            gradient_accumulation_steps=4,
            report_to="none",
            max_length=512,
            max_prompt_length=128,
        )

        # FIX 4: use processing_class= instead of deprecated tokenizer=
        self.trainer = TRLDPOTrainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset,
            processing_class=self.tokenizer
        )

        print(f"Training on {len(pairs)} pairs for {num_epochs} epochs...")

        # FIX 5: .train() returns a TrainOutput object, not a generator
        train_result = self.trainer.train()
        # FIX 6: Extract actual loss from the trainer's log history
        loss = 0.0
        if hasattr(self.trainer, 'state') and self.trainer.state.log_history:
            last_log = self.trainer.state.log_history[-1]
            loss = last_log.get('train_loss', last_log.get('loss', 0.0))
        if loss == 0.0:
            loss = getattr(train_result, 'training_loss', 0.0)

        self.stats["iterations_completed"] += 1
        self.stats["total_pairs_trained"] += len(pairs)
        if loss < self.stats["best_loss"]:
            self.stats["best_loss"] = loss

        return {
            "loss": loss,
            "pairs_trained": len(pairs),
            "epochs": num_epochs
        }

    def save_adapter(self, path: Path = None) -> Path:
        """Save the trained LoRA adapter."""
        # FIX 7: Save directly to self.output_dir so setup_model() can find it for resume
        path = Path(path or self.output_dir)
        path.mkdir(parents=True, exist_ok=True)

        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)

        with open(path / "training_stats.json", 'w') as f:
            json.dump(self.stats, f, indent=2)

        print(f"Saved adapter to {path}")
        return path

    def get_stats(self) -> Dict[str, Any]:
        """Get training statistics."""
        return self.stats.copy()


def create_dpo_pairs_from_results(
    winners: List[Dict[str, Any]],
    losers: List[Dict[str, Any]],
    prompt_template: str
) -> List[DPODataPoint]:
    """
    Create DPO training pairs from winner/loser results.

    Args:
        winners: List of winner test dicts with 'input', 'function_id'
        losers: List of loser test dicts
        prompt_template: Template string with {function_id} placeholder

    Returns:
        List of DPODataPoint objects
    """
    pairs = []

    for winner in winners:
        for loser in losers:
            if winner.get("function_id") == loser.get("function_id"):
                prompt = prompt_template.format(
                    function_id=winner.get("function_id", "unknown")
                )
                pairs.append(DPODataPoint(
                    prompt=prompt,
                    chosen=winner.get("input", ""),
                    rejected=loser.get("input", ""),
                    function_id=winner.get("function_id", "")
                ))

    return pairs

Overwriting /content/drive/MyDrive/Capstone/oneiros/engine/dpo_trainer.py


In [ ]:
%%writefile /content/drive/MyDrive/Capstone/oneiros/oneiros_loop.py

"""
Oneiros Engine - Main Learning Loop
"""
import sys
import json
import time
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

PROJECT_ROOT = Path(__file__).parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    dataset_config,
    model_config,
    memory_config,
    training_config,
    DATA_DIR,
    get_training_functions,
    get_testing_functions,
)

from harness.execution_harness import ExecutionHarness, ExecutionResult
from harness.system_dataset_loader import SystemLevelDatasetLoader

ENGINE_AVAILABLE = True
try:
    from engine import (
        FAISSMemory,
        Phi3Generator,
        MockGenerator,
        FeedbackOracle,
        DPOTrainer,
        DPODataPoint,
    )
except ImportError as e:
    ENGINE_AVAILABLE = False
    print(f"Warning: Engine not fully available: {e}")


@dataclass
class LoopConfig:
    """Configuration for the Oneiros learning loop."""
    num_iterations: int = 10
    tests_per_iteration: int = 16
    dpo_train_every: int = 5
    save_every: int = 2
    use_mock_generator: bool = True
    verbose: bool = True
    resume: bool = True          # NEW: auto-resume from latest checkpoint


class OneirosLoop:
    def __init__(self, config: LoopConfig = None):
        self.config = config or LoopConfig()

        print("Loading system-level functions...")
        self.training_functions = get_training_functions()
        self.testing_functions = get_testing_functions()
        print(f"  Training: {len(self.training_functions)} functions")
        print(f"  Testing: {len(self.testing_functions)} functions")

        self.harness = ExecutionHarness(timeout_seconds=5.0)
        self.memory = None
        self.generator = None
        self.oracle = None
        self.trainer = None

        self.stats = {
            "iterations": 0,
            "total_tests_generated": 0,
            "total_bugs_found": 0,
            "total_winners": 0,
            "total_losers": 0,
            "dpo_trainings": 0
        }

        self.all_winners = []
        self.all_losers = []
        self.start_iteration = 0   # NEW: tracks where to resume from

    # ------------------------------------------------------------------ #
    #  NEW: checkpoint detection                                           #
    # ------------------------------------------------------------------ #
    def find_latest_checkpoint(self) -> Optional[int]:
        """Return the highest saved iteration number, or None."""
        checkpoint_root = DATA_DIR / "checkpoints"
        if not checkpoint_root.exists():
            return None

        iterations = []
        for d in checkpoint_root.iterdir():
            if d.is_dir() and d.name.startswith("iter_"):
                try:
                    iterations.append(int(d.name.split("_")[1]))
                except ValueError:
                    pass

        return max(iterations) if iterations else None

    def load_checkpoint(self, iteration: int) -> None:
        checkpoint_dir = DATA_DIR / "checkpoints" / f"iter_{iteration}"

        stats_file = checkpoint_dir / "stats.json"
        if stats_file.exists():
            with open(stats_file) as f:
                self.stats = json.load(f)
            print(f"  ✓ Loaded stats from iter_{iteration}")

        # NEW: restore winners and losers
        winners_file = checkpoint_dir / "winners.json"
        losers_file = checkpoint_dir / "losers.json"
        if winners_file.exists() and losers_file.exists():
            with open(winners_file) as f:
                self.all_winners = json.load(f)
            with open(losers_file) as f:
                self.all_losers = json.load(f)
            print(f"  ✓ Loaded {len(self.all_winners)} winners, {len(self.all_losers)} losers")
        else:
            print(f"  ⚠ No winners/losers found in checkpoint — DPO will only use new data")

        memory_dir = checkpoint_dir / "memory"
        if memory_dir.exists() and self.memory:
            self.memory.load(memory_dir)
            print(f"  ✓ Loaded FAISS memory from iter_{iteration}")

        self.start_iteration = iteration
        print(f"  ✓ Resuming from iteration {iteration + 1}")

    def setup(self) -> None:
        """Initialize all components."""
        print("\nSetting up Oneiros Engine...")

        if not ENGINE_AVAILABLE:
            print("  Warning: Using mock components due to missing dependencies")
            self.config.use_mock_generator = True

        # Initialize FAISS Memory first (needed before load_checkpoint)
        try:
            self.memory = FAISSMemory()
            print("  ✓ FAISS Memory initialized")
        except Exception as e:
            print(f"  ✗ FAISS Memory failed: {e}")
            self.memory = None

        # Initialize Generator
        if self.config.use_mock_generator:
            self.generator = MockGenerator()
            print("  ✓ Mock Generator initialized")
        else:
            try:
                self.generator = Phi3Generator()
                print("  ✓ Phi-3 Generator initialized")
            except Exception as e:
                print(f"  ✗ Phi-3 failed, using mock: {e}")
                self.generator = MockGenerator()

        # Initialize Oracle
        if self.memory:
            self.oracle = FeedbackOracle(self.memory)
            print("  ✓ Feedback Oracle initialized")

            # Only seed memory if NOT resuming from checkpoint
            latest = self.find_latest_checkpoint() if self.config.resume else None

            if latest is not None:
                # Resume — load checkpoint instead of seeding fresh
                self.load_checkpoint(latest)
            else:
                # Fresh start — seed memory
                from harness.execution_harness import create_seed_tests
                seed_limit = min(100, len(self.training_functions))
                seed_tests = create_seed_tests(self.training_functions[:seed_limit])
                for t in seed_tests:
                    self.memory.add(t.code, t.target_function, found_bug=False)
                print(f"  ✓ Seed memory initialized with {len(seed_tests)} basic tests")
        else:
            print("  ✗ Oracle requires memory")

        # Initialize DPO Trainer
        if not self.config.use_mock_generator:
            try:
                self.trainer = DPOTrainer()
                print("  ✓ DPO Trainer initialized")
            except Exception as e:
                print(f"  ✗ DPO Trainer failed: {e}")

        print("\nSetup complete!")

    def _generate_tests(self, func, num_tests: int) -> List[Dict[str, Any]]:
        memory_examples = []
        if self.memory:
            memory_examples = self.memory.get_for_prompt(func.id, k=3)

        generated = self.generator.generate(
            function_signature=func.signature,
            docstring=func.docstring,
            function_id=func.id,
            edge_cases=func.edge_cases,
            memory_examples=memory_examples,
            library=func.library,
            num_samples=num_tests
        )

        return [{
            "id": g.id,
            "input": g.input_code,
            "function_id": g.function_id,
            "is_valid": g.is_valid
        } for g in generated]

    def _execute_test(self, test: Dict[str, Any], func) -> Dict[str, Any]:
        wrapper_code = func.wrapper_code
        result = self.harness.execute_test(
            test_code=test["input"],
            function_code=wrapper_code,
            entry_point=func.signature.split('(')[0].split()[-1],
            test_id=test["id"]
        )
        found_bug = result.result.value in ["fail", "error"]
        return {
            **test,
            "found_bug": found_bug,
            "execution_result": result.result.value,
            "error_message": result.error_message
        }

    def _evaluate_and_store(self, executed_tests: List[Dict[str, Any]]) -> None:
        if not self.oracle:
            return

        for test in executed_tests:
            result = self.oracle.evaluate(
                test_input=test["input"],
                found_bug=test.get("found_bug", False),
                is_valid=test.get("is_valid", True),
                function_id=test.get("function_id", "")
            )

            if result.is_winner():
                self.all_winners.append(test)
                self.stats["total_winners"] += 1
                if self.memory:
                    self.memory.add(
                        test_input=test["input"],
                        function_id=test["function_id"],
                        found_bug=test.get("found_bug", False)
                    )
                if test.get("found_bug"):
                    self.stats["total_bugs_found"] += 1
            else:
                self.all_losers.append(test)
                self.stats["total_losers"] += 1

    def run_iteration(self, iteration: int) -> Dict[str, Any]:
        if self.config.verbose:
            print(f"\n{'='*50}")
            print(f"Iteration {iteration + 1}/{self.config.num_iterations}")
            print('='*50)

        iteration_stats = {
            "iteration": iteration + 1,
            "tests_generated": 0,
            "bugs_found": 0,
            "winners": 0,
            "losers": 0
        }

        for func in self.training_functions[:10]:
            if self.config.verbose:
                print(f"\n  Testing: {func.name}")

            tests = self._generate_tests(func, num_tests=self.config.tests_per_iteration)
            iteration_stats["tests_generated"] += len(tests)
            self.stats["total_tests_generated"] += len(tests)

            if self.config.verbose:
                print(f"    Generated {len(tests)} tests")

            executed = []
            for test in tests:
                result = self._execute_test(test, func)
                executed.append(result)
                if result.get("found_bug"):
                    iteration_stats["bugs_found"] += 1

            self._evaluate_and_store(executed)

        iteration_stats["winners"] = self.stats["total_winners"]
        iteration_stats["losers"] = self.stats["total_losers"]
        self.stats["iterations"] = iteration + 1

        if self.config.verbose:
            print(f"\n  Results: {iteration_stats['bugs_found']} bugs, "
                  f"{self.stats['total_winners']} winners, "
                  f"{self.stats['total_losers']} losers")

        return iteration_stats

    def run(self) -> Dict[str, Any]:
        print("\n" + "="*60)
        print("ONEIROS ENGINE - Starting Learning Loop")
        print("="*60)

        self.setup()

        # NEW: if resuming, only run remaining iterations
        remaining = self.config.num_iterations - self.start_iteration
        if self.start_iteration > 0:
            print(f"\n  Resuming from iteration {self.start_iteration + 1} "
                  f"({remaining} iterations remaining)")

        start_time = time.time()

        for i in range(self.start_iteration, self.config.num_iterations):
            self.run_iteration(i)

            # DPO Training
            if (i + 1) % self.config.dpo_train_every == 0 and self.trainer:
                if self.config.verbose:
                    print(f"\n  [Triggering DPO Training (Iteration {i+1})]")

                recent_winners = self.all_winners[-self.config.tests_per_iteration * 6:]
                recent_losers = self.all_losers[-self.config.tests_per_iteration * 6:]

                if len(recent_winners) > 0 and len(recent_losers) > 0:
                    from engine.dpo_trainer import create_dpo_pairs_from_results
                    pairs = create_dpo_pairs_from_results(
                        winners=recent_winners,
                        losers=recent_losers,
                        prompt_template="Generate test for {function_id}"
                    )

                    if len(pairs) >= 4:
                        try:
                            print(f"    -> Training on {len(pairs)} preference pairs...")
                            train_config = self.trainer.train(pairs, num_epochs=1, batch_size=min(4, len(pairs)))
                            adapter_path = self.trainer.save_adapter()
                            self.generator.load_lora_adapter(adapter_path)
                            self.stats["dpo_trainings"] += 1
                            print(f"    -> DPO complete! Loss: {train_config.get('loss', 'N/A')}")
                        except Exception as e:
                            print(f"    -> DPO failed: {e}")
                            import traceback
                            traceback.print_exc()
                    else:
                        print(f"    -> Only {len(pairs)} pairs, need 4+ to train")
                else:
                    print("    -> Not enough winners/losers for DPO")

            if (i + 1) % self.config.save_every == 0:
                self.save_checkpoint(i + 1)

        elapsed = time.time() - start_time

        print("\n" + "="*60)
        print("ONEIROS ENGINE - Loop Complete")
        print("="*60)
        print(f"\nFinal Statistics:")
        print(f"  Iterations: {self.stats['iterations']}")
        print(f"  Tests Generated: {self.stats['total_tests_generated']}")
        print(f"  Bugs Found: {self.stats['total_bugs_found']}")
        print(f"  Winners: {self.stats['total_winners']}")
        print(f"  Losers: {self.stats['total_losers']}")
        print(f"  DPO Trainings: {self.stats['dpo_trainings']}")
        print(f"  Time: {elapsed:.2f}s")

        return self.stats

    def save_checkpoint(self, iteration: int) -> None:
        checkpoint_dir = DATA_DIR / "checkpoints" / f"iter_{iteration}"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        with open(checkpoint_dir / "stats.json", 'w') as f:
            json.dump(self.stats, f, indent=2)

        # NEW: save winners and losers too
        with open(checkpoint_dir / "winners.json", 'w') as f:
            json.dump(self.all_winners, f, indent=2)
        with open(checkpoint_dir / "losers.json", 'w') as f:
            json.dump(self.all_losers, f, indent=2)

        if self.memory:
            self.memory.save(checkpoint_dir / "memory")

        print(f"  [Checkpoint saved at iteration {iteration}]")


def main():
    config = LoopConfig(
        num_iterations=8,         # 6 done + 2 new = 8 total
        tests_per_iteration=8,
        dpo_train_every=2,
        save_every=2,
        use_mock_generator=False,
        verbose=True,
        resume=True               # will auto-detect iter_6 and skip to iter 7
    )

    loop = OneirosLoop(config)
    results = loop.run()
    return results


if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/Capstone/oneiros/oneiros_loop.py


In [ ]:
!python /content/drive/MyDrive/Capstone/oneiros/scripts/train_on_dataset.py

Loading train split ...
  Train pairs: 8,000
2026-04-06 13:26:28.518098: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775481988.553055    7021 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775481988.566644    7021 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775481988.595357    7021 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775481988.595403    7021 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775481988.595412    7021 comp